<div align="right"><sub>Notebook 最終更新: 2026-03-23 20:26</sub></div>
<h1><strong>07. RAG 統合マルチエージェント（最終回）</strong></h1>

演習の締めくくりとして，これまでに学んだ **RAG (外部知識の参照)** と **AIエージェント (Executor & Critic)** を統合します．
LLMが持っていない知識（2027年のアニメニュース）について回答し，その内容が検索結果と矛盾していないかを Critic が検証する，高度な信頼性を持ったシステムを構築しましょう．

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
print('永続ディレクトリ:', PERSIST_ROOT)
from src.common import load_llm, generate_text
from src.rag import RagEngine
from src.agent_core import LLMExecutorCriticAgent, RoleConfig
from src.ui import create_agent_ui

model, tokenizer = load_llm()
print('準備完了')


## **1. RAG統合エージェントの設定**
Executor には「RAG情報を元に回答する」役割を，Critic には「回答がRAG情報と矛盾していないかチェックする」役割を，それぞれプロンプトで与えます．

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

# RAGの初期化
rag = RagEngine()
index_path = os.path.join(PERSIST_INDEX_DIR, 'faiss.index')
chunks_path = os.path.join(PERSIST_INDEX_DIR, 'chunks.json')

if os.path.exists(index_path) and os.path.exists(chunks_path):
    rag.load_index(index_path, chunks_path)
else:
    rag.load_documents('data/docs/anime_docs_sample.jsonl')
    rag.build_index()
    os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)
    rag.save_index(index_path, chunks_path)

executor_config = RoleConfig(
    name='Fact Finder',
    system_prompt='あなたは優秀な調査官です。提供された資料に基づいて、質問に正確に回答してください。資料にないことは「わかりません」と答えてください。'
)
critic_config = RoleConfig(
    name='Fact Checker',
    system_prompt='あなたは厳格なファクトチェッカーです。回答が提供された資料と矛盾していないか、また資料にないことを勝手に話していないか厳しくチェックしてください。'
)

agent = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_config, critic_config])
print('準備完了')
print(f'RAGインデックス: {index_path}')


## **2. 統合UIの起動**
質問を入力すると，「RAG検索 → Executor回答 → Critic検証 → 修正回答」のフルプロセスが実行されます．

### 試してみるクエリの例（コピペ用）
1. **資料に基づいた正確な事実確認**:
   > 「2027年公開予定のSFアニメ『星空のレクイエム』のあらすじと，舞台となる都市の名前を資料から教えてください．」
2. **資料にない情報のハルシネーション（嘘）を誘発**:
   > 「『タイムリープ・カフェ』の主題歌を歌っているアーティストは誰ですか？（資料にない情報をExecutorが捏造し，Criticが指摘するか確認）」
3. **複数の情報をまとめる**:
   > 「2027年公開のアニメ作品について，資料から判明している作品タイトルと監督名を一覧にしてください．」

In [ ]:
def run_rag_agent(query):
    # 1. RAG検索
    results = rag.search(query, top_k=3)
    context = ""
    for i, res in enumerate(results):
        context += f"[資料{i+1}] {res['chunk']['text']}\n"
    
    # 2. エージェントへの入力構築
    agent_input = f"【資料】\n{context}\n\n【質問】\n{query}"
    
    # 3. エージェント実行
    final_answer, full_log, steps = agent.run_pipeline(agent_input)
    
    formatted_log = f"### RAG 検索結果\n"
    for i, res in enumerate(results):
        formatted_log += f"* {res['chunk']['title']}: {res['chunk']['text'][:50]}...\n"
    
    formatted_log += "\n--- AIエージェントの処理過程 ---\n"
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
        
    return final_answer, formatted_log

ui = create_agent_ui(run_rag_agent)
ui.launch(share=True)

## **演習のまとめ（コース修了）**

このコースでは，最新の日本語LLM **Qwen 2.5** を使って、以下のステップを学びました：

1. **LLMの基礎**: 生成の仕組みとハルシネーションの限界
2. **プロンプト制御**: 指示による振る舞いの変化
3. **RAG (検索拡張生成)**: 外部の膨大な知識をLLMに「読ませる」方法
4. **AIエージェント**: 複数の役割を組み合わせた「考えるシステム」の構築

単体では不完全なLLMも，RAGやエージェントといった「周辺技術」と組み合わせることで，非常に信頼性の高い，実用的で美しいシステムへと進化させることができます．ここでの学びを，ぜひ独自のAIアプリケーション開発に活かしてください！